In [ ]:
import json
import pandas as pd

In [ ]:
with open("/mnt/data/upcast/data/arxpr_simplified.json", "r") as f:
    json_file = json.load(f)
print(len(json_file))

In [ ]:
for key in json_file:
    fields = list(json_file[key].keys())
    break
fields


In [ ]:
dfs = {}
for field in fields:
    df = []

    for key in json_file:
        for element in json_file[key][field]:
            df.append({
                "id": key,
                "value" : element["value"],
                "ont" : None if element["ontology"] is None else element["ontology"][0],
                "ont_term" : None if element["ontology"] is None else element["ontology"][1],
                })
    df = pd.DataFrame(df)
    if not df.nunique()["ont_term"]:
        df = df[["id", "value"]]
    dfs[field] = df

In [ ]:
from collections import Counter
from matplotlib import pyplot as plt

In [ ]:
for field, n in [
    ("experimental_factors_20", 10),
    ("study_type_18", 25),
    ("type_21", 50),
]:
    commons = Counter(dfs[field]["value"]).most_common(n)
    print([c[0] for c in commons])

In [ ]:

for field in dfs:
    df = dfs[field]
    commons = Counter(df["value"]).most_common(20)
    commons = {t[0]:t[1] for t in commons}
    commons["rest"] = len(df)-sum(commons.values())
    plt.bar(commons.keys(), commons.values())
    plt.xticks(rotation=90)
    plt.title(label=field)

    plt.show()

    commons = Counter(df["value"]).most_common()
    f = len(commons)//30+1

    print("Field name:", field)
    print("Select values:", commons[::f])
    print("Number of values:", len(df))
    print(df.rename(columns={
            "id":"number of datasets with this field:",
            "value":"number of unique values:",
            "ont": "number of ontologies refered:",
            "ont_term":"number of unique ontology terms:"
            }).nunique())
    print("###"*50)


# print values for literals

In [ ]:
print("values = dict(")
for field in [
    #'sex_2',
    'hardware_4',
    'organism_part_5',
    'experimental_designs_10',
    'assay_by_molecule_14',
    'study_type_18',
    #'experimental_factors_20',
    #'type_21',
    #'adjusted_type_24'
    ]:
    df = dfs[field]
    commons = Counter(df["value"]).most_common()
    commons = [c[0] for c in commons]
    print(f"    {field} = dict(")
    print(f"        _25=", commons[:25],",")
    print(f"        _50=", commons[25:50],",")
    print(f"        _100=", commons[50:100],",")
    print(f"        _200=", commons[100:200],",")
    #print(f"        _400=", commons[200:400],",")
    print("    ),")
print(")")



# another distribution visualizing method
number of labels as fnc of number of different labels

In [ ]:

for field in [
    #'sex_2',
    'hardware_4',
    'organism_part_5',
    'experimental_designs_10',
    'assay_by_molecule_14',
    'study_type_18',
    #'experimental_factors_20',
    #'type_21',
    #'adjusted_type_24'
    ]:

    df = dfs[field]

    commons = Counter(df["value"]).most_common()

    x = []
    y = []
    accumulative_count = 0
    i = 1
    for name, number in commons:
        accumulative_count += number
        x.append(i)
        y.append(accumulative_count)
        i += 1
    
    plt.plot(x,y)
    plt.grid()
    plt.title(label=field+f", first 25: {y[min(len(y)-1, 24)]}"+f", first 50: {y[min(len(y)-1, 49)]}")
    #plt.xlim(0,250)
    plt.show()

    df = dfs[field]
    commons = Counter(df["value"]).most_common(25)
    commons = {t[0]:t[1] for t in commons}
    commons["rest"] = len(df)-sum(commons.values())
    plt.bar(commons.keys(), commons.values())
    plt.xticks(rotation=90)
    plt.title(label=field)

    plt.show()

    commons = Counter(df["value"]).most_common()
    f = len(commons)//30+1

    print("Field name:", field)
    print("Select values:", commons[::f])
    print("Number of values:", len(df))
    print(df.rename(columns={
            "id":"number of datasets with this field:",
            "value":"number of unique values:",
            "ont": "number of ontologies refered:",
            "ont_term":"number of unique ontology terms:"
            }).nunique())
    print("###"*50)



# ontology fields

In [ ]:

for field in dfs:
    df = dfs[field]


    if len(df.columns) > 2:
        print("Field name:", field)
        print("Number of values:", len(df))
        print(df.dropna(subset="ont_term").rename(columns={
                "id":"number of datasets with ontology term:",
                "value":"number of unique values with ontology term:",
                "ont": "number of ontologies refered:",
                "ont_term":"number of unique ontology terms:"
                }).nunique())
        print(df.dropna(subset="ont_term").drop("id", axis=1).drop_duplicates(subset = ["value", "ont_term"]))


## not one-to-one relationship between value and ontology term

In [ ]:
uniques_21 = dfs["type_21"].dropna(subset="ont_term").drop_duplicates(subset = ["value", "ont_term"])
uniques_21

In [ ]:
# diferent ont_terms for same value
uniques_21[uniques_21["value"] == "growth protocol"]

In [ ]:
# different value for same ont_term
uniques_21[uniques_21["ont_term"] == "efo_0003789"]

In [ ]:
# ont_term specified with eiter ":" or "_"
pd.concat([
    uniques_21[uniques_21["ont_term"] == "efo_0003789"],
    uniques_21[uniques_21["ont_term"] == "efo:0003789"]
])
    


## plot distribution, like before, but only datasets with ontology terms

In [ ]:

for field in dfs:
    if len(dfs[field].columns) > 2:

        df = dfs[field].dropna(subset="ont_term")
        commons = Counter(df["value"]).most_common(60)
        commons = {t[0]:t[1] for t in commons}
        #commons["rest"] = len(df)-sum(commons.values())
        plt.bar(commons.keys(), commons.values())
        plt.xticks(rotation=90)

        plt.show()

        commons = Counter(df["value"]).most_common()
        f = len(commons)//30+1

        print("Field name:", field)
        print("ALL values", commons)
        print("Number of values:", len(df))
        print(df.rename(columns={
                    "id":"number of datasets with ontology term:",
                    "value":"number of unique values with ontology term:",
                    "ont": "number of ontologies refered:",
                    "ont_term":"number of unique ontology terms:"
                }).nunique())


In [ ]:
# print ont terms for use in the subtre_investigtion notebook

dfs["study_type_18"].dropna(subset="ont_term").drop_duplicates(subset = ["value", "ont_term"])["ont_term"].values


## final dataset

In [ ]:
with open("/mnt/data/upcast/data/arxpr2_25_metadataset_restricted_values.json", "r") as f:
    json_file = json.load(f)
print(len(json_file))

In [ ]:
json_file